# 08 — Kombinierte Query: Behörden + News (ohne Journalist:innen)

Liest [`data/accounts.csv`](../data/accounts.csv) und baut **eine** Brandwatch-Query,
die die Inhalte aus Notebook 05 (Behörden) und Notebook 07 (News ohne Journalist:innen)
per `OR` zusammenführt.

**Filter (Vereinigung beider Teile):**
- Teil A — Behörden: `category == "Organisation"` AND `label == "Behörde"`
- Teil B — News: `category == "News"` AND `label != "Journalist"`
- Beide: `channel ∈ {x, instagram, facebook}`

**Struktur:** ein Behörden-Block + ein Block pro News-Label. Labels alphabetisch,
Handles alphabetisch, Handles pro Block auf einer Zeile.

**Output:** `output/queries/behoerden_news_query.txt`.

⚠️ ~3900 Handles — die Query kann das 100k-Limit reißen. Bei Bedarf News-Labels
(z. B. *Entertainment*, *Zeitung*) aus der TXT wegschneiden.

In [1]:
import os

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR  = os.path.join(PROJECT_ROOT, "output", "queries")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE  = os.path.join(QUERIES_DIR, "behoerden_news_query.txt")

ALLOWED_CHANNELS = ["x", "instagram", "facebook"]
LANGUAGE_FILTER  = "language:de"

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + beide Teile filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)

def prep(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df[df["channel"].isin(ALLOWED_CHANNELS)]
        .dropna(subset=["handle"])
        .drop_duplicates(subset=["channel", "handle"])
        .copy()
    )

behoerden = prep(
    accounts[(accounts["category"] == "Organisation") & (accounts["label"] == "Behörde")]
)
news = prep(
    accounts[(accounts["category"] == "News") & (accounts["label"] != "Journalist")]
    .dropna(subset=["label"])
)

print(f"Behörden: {len(behoerden):>5}")
print(f"News:     {len(news):>5}")
print(f"Gesamt:   {len(behoerden) + len(news):>5}")
print()
print("News-Labels:")
print(news["label"].value_counts())

Behörden:   595
News:      3297
Gesamt:    3892

News-Labels:
label
Rundfunksender         1298
Zeitung                 998
Online_Only             543
Entertainment           331
Nachrichtenprogramm      97
Nachrichtenagentur       30
Name: count, dtype: int64


## 2. Helper

In [3]:
def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def block(comment: str, handles) -> str:
    body = " OR ".join(bw_author(h) for h in handles)
    return f"<<< {comment} — {len(handles)} Handles >>>\n({body})"


def sort_handles(series: pd.Series) -> list[str]:
    return series.sort_values(key=lambda s: s.str.lower()).tolist()

## 3. Blöcke bauen

In [4]:
blocks: list[str] = []

# --- Teil A: Behörden (ein Block) ---
if not behoerden.empty:
    blocks.append(block("Behörden", sort_handles(behoerden["handle"])))
    print(f"Behörden: 1 Block ({len(behoerden)} Handles)")

# --- Teil B: News (pro Label ein Block, alphabetisch) ---
news_label_count = 0
for label in sorted(news["label"].unique(), key=str.lower):
    handles = sort_handles(news[news["label"] == label]["handle"])
    blocks.append(block(f"News — {label}", handles))
    news_label_count += 1
print(f"News:     {news_label_count} Blöcke ({len(news)} Handles)")

print(f"\nBlöcke gesamt: {len(blocks)}")

Behörden: 1 Block (595 Handles)
News:     6 Blöcke (3297 Handles)

Blöcke gesamt: 7


## 4. Gesamt-Query zusammensetzen

In [5]:
total_handles = sum(b.count('author:"') for b in blocks)
header = f"<<< Behörden + News — {total_handles} Handles in {len(blocks)} Blöcken >>>"

body = "\nOR\n".join(blocks)
query = f"{header}\n({LANGUAGE_FILTER} AND (\n{body}\n))\n"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

size = os.path.getsize(OUTPUT_FILE)
print(f"Datei:           {OUTPUT_FILE}")
print(f"Größe:           {size:,} bytes  ({size / 1024:.1f} KiB)")
print(f"Handles gesamt:  {total_handles}")
print(f"Blöcke:          {len(blocks)}")

if size > 100_000:
    print(f"\n⚠️  {size - 100_000:,} Zeichen über dem 100k-Limit.")

Datei:           /Users/zorbeyozcan/Projekte/query_printer/output/queries/behoerden_news_query.txt
Größe:           100,477 bytes  (98.1 KiB)
Handles gesamt:  3892
Blöcke:          7

⚠️  477 Zeichen über dem 100k-Limit.


## 5. Preview

In [6]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if "<<<" in line and ">>>" in line:
            print(line.rstrip())

<<< Behörden + News — 3892 Handles in 7 Blöcken >>>
<<< Behörden — 595 Handles >>>
<<< News — Entertainment — 331 Handles >>>
<<< News — Nachrichtenagentur — 30 Handles >>>
<<< News — Nachrichtenprogramm — 97 Handles >>>
<<< News — Online_Only — 543 Handles >>>
<<< News — Rundfunksender — 1298 Handles >>>
<<< News — Zeitung — 998 Handles >>>
